# Site feasibility, end to end

## AstraZeneca hands-on lab

Before a clinical study can open at an investigator site, somebody has to read that
site's feasibility survey and work out whether the site can actually run the
protocol. Can they do a cardiac MRI on site? Do they have a minus 70 freezer? Is
there an on-site pharmacy?

These surveys arrive as PDFs. Someone reads each one by hand, cross-references the
protocol requirements, and types the answers into a spreadsheet. For a study with a
few hundred candidate sites that is weeks of work, and the spreadsheet is stale the
moment a new survey arrives.

In the next hour you will replace that with a pipeline.

### What you will build

| Step | What happens |
|---|---|
| 1 | Extract structured capability data straight out of 10 survey PDFs |
| 2 | Repair the messy names, derive the answers, and score against ground truth |
| 3 | Parse the same PDFs to text, chunk them, and build a search service |
| 4 | Expose the numbers through a semantic view, askable in English |
| 5 | Use an agent with three tools, including one that writes and runs Python |
| 6 | Deploy an app on Snowpark Container Services and change how it ranks sites |

Two halves worth noticing as you go. Steps 1 and 2 turn the documents into **numbers**.
Step 3 keeps the **words**. Steps 4 to 6 are about asking questions of both, because
most real questions need the number and the reason behind it.

### How this works

Every cell is pre-written. Click the play button, read the result, and listen to
the explanation. There is nothing to edit and nothing you can break.

Each step ends with a **checkpoint** that prints PASS so you know the step
worked before moving on.

Everything is synthetic. Every site, investigator and study code is fictional.

### Set your role and warehouse first

At the top of this notebook there are two selectors: **Choose role** and
**Choose warehouse**. Set them to **`ACCOUNTADMIN`** and **`AZ_HOL_WH`** before you
run anything.

You are ACCOUNTADMIN in this account, which is not how you would run this in
production - there you would grant a purpose-built role exactly what the pipeline
needs and nothing more. For a one-hour lab it removes a whole class of permission
detour that teaches you nothing about the platform.

The first cell reports what you actually have, so you will know either way.

In [ ]:
-- RUN. Reports your session and turns off result caching so that every query below
-- does genuine work rather than replaying a cached answer.
--
-- Role and warehouse come from the selectors at the top of the notebook, not from
-- this cell. That is deliberate: hardcoding a role here would break the moment your
-- account was provisioned with a different one.
ALTER SESSION SET USE_CACHED_RESULT = FALSE;

USE DATABASE AZ_FEASIBILITY_DEMO;
USE SCHEMA EXTRACTED;

SELECT CURRENT_ROLE()      AS your_role,
       CURRENT_WAREHOUSE() AS your_warehouse,
       CURRENT_DATABASE()  AS database,
       CURRENT_REGION()    AS region;

### The documents are already here

Nobody has to upload anything. The 10 surveys were placed on a stage in **your**
account when it was provisioned.

One thing worth knowing, because it is the most common way this breaks: the stage
must use **server-side encryption**. Internal stages default to client-side
encryption, and a client-side encrypted PDF is unreadable to `AI_EXTRACT`. The
upload succeeds, `LIST` shows the file, and extraction silently returns nothing.

```sql
CREATE STAGE ... ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
```

The encryption type cannot be changed after the stage is created.

### What a stage is

A **stage** is where files live inside Snowflake. Think of it as a folder that
Snowflake can read directly - no separate object store to configure, no
credentials to pass around, and the same access controls as everything else.

The 10 survey PDFs are already on a stage called `RAW.SURVEYS`. The cell below
lists them.

One detail that matters more than it looks: the stage was created with
`ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')`. Internal stages default to *client-side*
encryption, and the vision model behind `AI_EXTRACT` cannot read a client-side
encrypted PDF. It does not error - it returns nothing. That failure mode is
silent, which makes it worth knowing about.

In [ ]:
-- RUN. The surveys already on your stage.
SELECT RELATIVE_PATH AS document,
       ROUND(SIZE / 1024, 1) AS size_kb,
       LAST_MODIFIED
FROM DIRECTORY(@AZ_FEASIBILITY_DEMO.RAW.SURVEYS)
ORDER BY RELATIVE_PATH;

In [ ]:
-- RUN. Ten seconds now saves ten minutes later.
--
-- Checks that your account was provisioned correctly before you depend on it.
-- Every row should say PASS. If any row says STOP, tell the facilitator rather
-- than trying to fix it - it means the account build did not finish, not that
-- you have done anything wrong.
SELECT 'Survey PDFs' AS what,
       (SELECT COUNT(*) FROM DIRECTORY(@AZ_FEASIBILITY_DEMO.RAW.SURVEYS)) AS found,
       10 AS expected,
       IFF((SELECT COUNT(*) FROM DIRECTORY(@AZ_FEASIBILITY_DEMO.RAW.SURVEYS)) = 10,
           'PASS', 'STOP - tell the facilitator') AS status
UNION ALL
SELECT 'Reference data CSVs',
       (SELECT COUNT(*) FROM DIRECTORY(@AZ_FEASIBILITY_DEMO.RAW.REFDATA)), 6,
       IFF((SELECT COUNT(*) FROM DIRECTORY(@AZ_FEASIBILITY_DEMO.RAW.REFDATA)) = 6,
           'PASS', 'STOP - tell the facilitator')
UNION ALL
SELECT 'Streamlit app files',
       (SELECT COUNT(*) FROM DIRECTORY(@AZ_FEASIBILITY_DEMO.STREAMLIT.APP_FILES)
         WHERE RELATIVE_PATH IN ('app.py', 'pyproject.toml')), 2,
       IFF((SELECT COUNT(*) FROM DIRECTORY(@AZ_FEASIBILITY_DEMO.STREAMLIT.APP_FILES)
             WHERE RELATIVE_PATH IN ('app.py', 'pyproject.toml')) = 2,
           'PASS', 'STOP - needed in step 6')
ORDER BY status DESC, what;

Open one of them. In the left-hand navigation go to **Data** then **Databases**,
then `AZ_FEASIBILITY_DEMO` → `RAW` → **Stages** → `SURVEYS`, and click a PDF.

Look at the site capabilities question. In some documents each item has a checkbox
that is either ticked or empty. In others the survey simply lists the items the site
has. Both are real formats, and the pipeline has to handle both.

---

# Step 1: Extract

## `AI_EXTRACT` reads the PDF directly

There is no parsing step. `AI_EXTRACT` takes a `FILE` and returns structured JSON:

```sql
AI_EXTRACT(file => TO_FILE('@stage', relative_path), responseFormat => {...})
```

That matters more than it looks. `AI_EXTRACT` runs on `arctic-extract`, a
vision model that reads the page image, so it sees a tick in a checkbox. If you
flattened the document to text first, the tick would become nothing at all -
exactly the signal you need, gone.

Two things about the schema below:

**It asks for a table, not two lists.** The first version of this lab asked for
`ticked_items` and `unticked_items` as separate arrays. The ticked list was perfect;
the unticked list pulled options in from a completely different question. Asking for
one row per item with a yes/no flag scored **8 of 8** where the two-list version
scored partially, and confidence went from 0.76 to **0.96**. Never ask a model for
the set of things that were *not* chosen.

**`scores => TRUE` returns a confidence per field**, free. You will use it in step 2.

### What AI_EXTRACT does

`AI_EXTRACT` reads a file and returns structured JSON. It is a vision model, so
it sees the page the way you do - including whether a checkbox is ticked, which
no text-only parser can tell you.

The important argument is **`responseFormat`**. You hand it a JSON schema
describing the fields you want, and those are the fields you get back. The schema
is the contract: change it and the output changes shape. Nothing is inferred
about what you might have wanted.

Two things worth watching for in the result:

- **`scores => TRUE`** returns a calibrated confidence per field, free. You do not
  have to guess how sure the model was.
- Everything comes back as a **string**, even numbers. That is deliberate - it
  means a messy answer like "about 20" is preserved rather than silently coerced
  to something wrong. Casting is your decision, later.

In [ ]:
-- RUN. Extracts from all 10 PDFs. Expect 1 to 2 minutes. This is the slowest cell
-- in the lab, because it is the only one that reads every page image of every
-- document. Everything after it works on the table this produces.
--
-- Read this cell as four moves:
--   1. DIRECTORY() at the bottom lists the PDFs on the stage. No file name is
--      hard coded, so a new survey landing on the stage is picked up next run.
--   2. TO_FILE() hands AI_EXTRACT the PDF itself, not text pulled out of it.
--      The model reads the page image, which is how it can see a tick.
--   3. responseFormat is the contract. Every field named below comes back and
--      nothing else. The descriptions ARE the instructions to the model.
--   4. scores => TRUE adds a confidence value alongside every field.
CREATE OR REPLACE TABLE AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_RAW AS
SELECT
  REGEXP_REPLACE(relative_path, '\\.pdf$', '') AS document_id,
  relative_path,
  AI_EXTRACT(
    -- A reference to the file on the stage, resolved at query time.
    file => TO_FILE('@AZ_FEASIBILITY_DEMO.RAW.SURVEYS', relative_path),
    responseFormat => {
      'schema': {
        'type': 'object',
        'properties': {
          -- The capability question. Returned as two parallel arrays, with
          -- column_ordering keeping them aligned so element 3 of item_name
          -- belongs with element 3 of is_available.
          'capabilities': {
            'description': 'The question asking what testing capabilities and equipment the site has. Return every item listed under that question. If each item has a checkbox, report whether it is ticked. If the question instead lists only the available items with no checkboxes, report YES for every item listed.',
            'type': 'object',
            'column_ordering': ['item_name', 'is_available'],
            'properties': {
              'item_name':    {'description': 'The exact text of the capability or equipment item as printed', 'type': 'array'},
              'is_available': {'description': 'YES if this item is available at the site, meaning its checkbox is ticked or it appears in the list of available items. NO if its checkbox is empty.', 'type': 'array'}
            }
          },
          'assessment_question': {'description': 'The exact wording of the question in the Study Assessments section', 'type': 'string'},
          'assessment_answer':   {'description': 'The answer given to the Study Assessments question. Return only Yes or No.', 'type': 'string'},
          'site_reference':      {'description': 'The Site Reference identifier, for example SITE_001', 'type': 'string'},
          'patients_seen_per_year': {'description': 'How many patients with the indication the site sees per year', 'type': 'string'},
          'planned_enrolled':    {'description': 'Patients enrolled per year', 'type': 'string'},
          'startup_weeks':       {'description': 'Estimated weeks from final protocol to first patient in', 'type': 'string'},
          -- The schema is yours to define. Adding a line here, for example
          -- 'pi_last_name', would return that field too - you are not limited to
          -- what someone else thought to ask for.
          'institution': {'description': 'The name of the institution, practice or clinic', 'type': 'string'}
        }
      }
    },
    scores => TRUE
  ) AS payload
FROM DIRECTORY(@AZ_FEASIBILITY_DEMO.RAW.SURVEYS);

SELECT COUNT(*) AS documents_processed FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_RAW;

### Look at what came back

One document, side by side with the PDF you opened earlier. Notice the confidence
score on each field.

In [ ]:
-- RUN. The raw JSON for one document.
SELECT document_id,
       payload:response:institution::STRING          AS institution,
       payload:response:site_reference::STRING       AS site_reference,
       payload:response:startup_weeks::STRING        AS startup_weeks,
       payload:response:assessment_answer::STRING    AS assessment_answer,
       ROUND(payload:scoring:scores:capabilities:score::FLOAT, 3) AS capabilities_confidence,
       payload:response:capabilities                 AS capabilities
FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_RAW
WHERE document_id = 'survey_08_site_008_d0000c00040';

### How the confidence score is actually calculated

The score is not a feeling. It is a measured quantity, and it is worth knowing how
it is produced before you decide how far to trust it.

**What you are using here is GA.** The model returns the numeric score **in the
same pass as the answer**, so there is no second call and no extra cost. Models tend
to be overconfident, so the raw value is **calibrated against measured accuracy**
before it is returned. That calibration is the reason the number tracks real hit
rates rather than the model's self-belief, and it is what makes it usable as a
threshold rather than as decoration.

**A different method is coming in private preview**, so if you read something
elsewhere that describes a second scoring call, that is the newer approach and not
what is running behind the cells in this lab. Nothing in this notebook depends on
which method produced the number.

**What it is good for.** It is a reliable signal for *ranking* which values to trust
- thresholds, fallbacks, and routing doubtful records to a human. Set your own cut by
labelling a sample of your own documents and measuring accuracy at a few different
cutoffs, so the threshold reflects your risk tolerance and your document mix rather
than a number taken from a lab.

**Evidence from this lab.** `AI_EXTRACT` was asked for a study code on a real survey
and returned `D576AC00003`. The correct answer was `D516AC00003` - it misread a 1 as a
7. It scored that field **0.32**, while every field it got right scored 0.65 or
higher. That is why the threshold later in this lab is 0.70 rather than a number
somebody picked, and it is why the study code here comes from the **filename** instead
of from extraction: it is a join key, and a silently misread key corrupts everything
downstream.

### Turning JSON into rows

The extraction returns one JSON object per document, with a nested array of
capability items inside it. That is awkward to query and impossible to join.

**`LATERAL FLATTEN`** unnests it: one row per array element, with the parent
columns carried alongside. It is the standard way to get from a nested response to
something you can treat as a table - which is what the rest of the lab needs.

In [ ]:
-- RUN. One row per capability item.
CREATE OR REPLACE TABLE AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_FLAT AS
WITH cols AS (
  SELECT document_id,
         payload:response:capabilities:item_name       AS names,
         payload:response:capabilities:is_available    AS flags,
         payload:response:assessment_question::STRING  AS assessment_question,
         payload:response:assessment_answer::STRING    AS assessment_answer,
         payload:response:site_reference::STRING       AS site_reference,
         payload:response:patients_seen_per_year::STRING AS patients_seen_per_year,
         payload:response:planned_enrolled::STRING     AS planned_enrolled,
         payload:response:startup_weeks::STRING        AS startup_weeks,
         payload:scoring:scores:capabilities:score::FLOAT AS capabilities_score
  FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_RAW
)
SELECT c.document_id,
       n.index                                    AS item_index,
       n.value::STRING                            AS raw_item_name,
       UPPER(TRIM(GET(c.flags, n.index)::STRING)) AS raw_is_available,
       c.assessment_question, c.assessment_answer, c.site_reference,
       c.patients_seen_per_year, c.planned_enrolled, c.startup_weeks,
       c.capabilities_score
FROM cols c, LATERAL FLATTEN(input => c.names) n;

SELECT document_id,
       COUNT(*)                            AS items,
       COUNT_IF(raw_is_available = 'YES')  AS available,
       COUNT_IF(raw_is_available = 'NO')   AS not_available,
       ROUND(MAX(capabilities_score), 3)   AS confidence
FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_FLAT
GROUP BY document_id ORDER BY document_id;

Look at the `not_available` column. Seven documents show zero, three show one or two.

That is not a bug. The three with explicit noes are the checkbox surveys, where an
empty box is a visible answer. The other seven simply **list what the site has** -
so an item the site lacks is absent from the page entirely.

Absence is the answer in those documents. Step 2 handles it.

**Checkpoint 1**

In [ ]:
-- RUN. Should print PASS.
SELECT CASE
         WHEN COUNT(DISTINCT document_id) = 10 AND COUNT(*) BETWEEN 100 AND 130
           THEN 'PASS - ' || COUNT(*) || ' items from ' || COUNT(DISTINCT document_id) || ' documents'
         ELSE 'CHECK - got ' || COUNT(*) || ' items from ' || COUNT(DISTINCT document_id) || ' documents'
       END AS checkpoint_1
FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_FLAT;

---

# Step 2: Normalise and score

## The names do not match anything

Site staff type free text. A controlled vocabulary says `CT Scan`; the survey says
`CT scan`. The vocabulary says `12-Lead ECG`; the survey says `12 Lead ECG`. One
survey in this set says `Tumor biopsy` where the taxonomy says `Tumour Biopsy` -
American spelling against British.

Match on exact strings and you lose a third of your data.

In [ ]:
-- RUN. Items whose printed name is not in the taxonomy at all.
SELECT f.raw_item_name AS written_on_survey,
       COUNT(*)        AS occurrences
FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_FLAT f
LEFT JOIN AZ_FEASIBILITY_DEMO.ANALYTICS.ITEM_TAXONOMY t
       ON t.canonical_name = f.raw_item_name
WHERE t.canonical_name IS NULL
GROUP BY 1 ORDER BY 2 DESC, 1;

### Fix it with one join, not a service

The obvious reach here is a search service with a reranker. That is three moving
parts, a wait for indexing, an LLM call per candidate, and a different answer on
different runs.

It is a fuzzy match against a known 40-row list. One SQL statement does it, the same
way every time:

```sql
JOIN taxonomy t ON VECTOR_COSINE_SIMILARITY(i.emb, t.emb) >= 0.85
QUALIFY ROW_NUMBER() OVER (PARTITION BY document_id, item_name
                           ORDER BY name_similarity DESC) = 1
```

`0.85` is not arbitrary. Measured on this vocabulary:

| Pair | Similarity |
|---|---|
| `CT scan` vs `CT Scan` | 0.984 |
| `12 Lead ECG` vs `12-Lead ECG` | 0.922 |
| `CT scan` vs `Centrifuge` | 0.382 |

Comfortable margin on both sides. The `QUALIFY` keeps exactly one canonical match
per raw item, which is what stops the join fanning out and double counting.

### What an embedding is

Site staff type free text. One survey says "12 Lead ECG", another
"Electrocardiogram (ECG)", a third just "ECG". All three mean the same thing, and
none of them match a controlled vocabulary exactly.

An **embedding** turns a piece of text into a list of numbers positioned so that
similar meanings land near each other. "Electrocardiogram" ends up close to "ECG"
because they are used in the same contexts, not because they share letters.

**Cosine similarity** measures how close two of those positions are, on a scale
where 1.0 is identical and 0 is unrelated. We accept a match at **0.85 or above**.
That threshold is a judgement: too low and you match things that only look
related, too high and genuine synonyms fall through and get recorded as missing
capability.

This is why the fix is a join and not a service. No API, no synonym list to
maintain by hand - the meaning comparison happens in SQL.

In [ ]:
-- RUN. Resolves every printed name to a canonical item.
--
-- 0.85 is the accept threshold. At 0.99 almost nothing matches and real synonyms
-- get recorded as missing capability. At 0.60 things that merely look related
-- start matching. 0.85 is where that trade-off sits for this vocabulary.
SET similarity_threshold = 0.85;

-- How the matching works, in three moves:
--   1. AI_EMBED turns a name into a vector - a list of numbers encoding what
--      the phrase MEANS. Close meanings land close together, so 'Cardiac MRI'
--      and 'MRI (Cardiac)' become near neighbours despite sharing almost no
--      characters. Both sides must use the same model or the vectors are not
--      comparable.
--   2. VECTOR_COSINE_SIMILARITY measures the angle between two vectors: 1.0 is
--      identical meaning, 0 is unrelated.
--   3. The JOIN keeps pairs above the threshold; QUALIFY keeps the single best
--      match per printed name, so one row in cannot become several rows out.
--
-- Note there is no synonym list anywhere in this cell. Nothing to maintain by
-- hand, and a name nobody has seen before still resolves.

CREATE OR REPLACE TABLE AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_NORMALISED AS
WITH extracted AS (
  -- The messy names, exactly as they were printed on the surveys.
  SELECT document_id, raw_item_name, raw_is_available, capabilities_score,
         AI_EMBED('snowflake-arctic-embed-l-v2.0', raw_item_name) AS emb
  FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_FLAT
),
taxonomy AS (
  -- The 40 approved names from the taxonomy, embedded with the same model.
  SELECT canonical_name, category,
         AI_EMBED('snowflake-arctic-embed-l-v2.0', canonical_name) AS emb
  FROM AZ_FEASIBILITY_DEMO.ANALYTICS.ITEM_TAXONOMY
)
SELECT e.document_id, e.raw_item_name, t.canonical_name, t.category,
       IFF(e.raw_is_available = 'YES', 'CAN', 'CANNOT') AS capability,
       e.capabilities_score AS confidence,
       VECTOR_COSINE_SIMILARITY(e.emb, t.emb) AS name_similarity
FROM extracted e
-- Joining on meaning, not on text. Every printed name is compared with every
-- approved name and only close pairs survive.
JOIN taxonomy t
  ON VECTOR_COSINE_SIMILARITY(e.emb, t.emb) >= $similarity_threshold
-- Several approved names can clear the bar for one printed name. Keep the
-- closest and drop the rest, so the count out matches the count in.
QUALIFY ROW_NUMBER() OVER (PARTITION BY e.document_id, e.raw_item_name
                           ORDER BY name_similarity DESC) = 1;

SELECT (SELECT COUNT(*) FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_FLAT) AS extracted,
       COUNT(*) AS matched,
       (SELECT COUNT(*) FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_FLAT) - COUNT(*) AS lost,
       COUNT_IF(raw_item_name <> canonical_name) AS repaired,
       ROUND(MIN(name_similarity), 3) AS weakest_match
FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_NORMALISED;

In [ ]:
-- RUN. What the join actually repaired.
SELECT raw_item_name AS written_on_survey,
       canonical_name AS resolved_to,
       ROUND(name_similarity, 3) AS similarity,
       COUNT(*) AS occurrences
FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_NORMALISED
WHERE raw_item_name <> canonical_name
GROUP BY ALL
ORDER BY similarity ASC;

### Question polarity

Most survey questions are direct: *"Can your site perform cardiac MRI?"* A ticked
**Yes** means the site can.

But some are phrased as problems: *"Would storing samples at minus 70 degrees present
a difficulty at your site?"* Here a ticked **Yes** means the site **cannot**.

Same word, opposite meaning. Get this wrong and the capability data is inverted for
every negatively phrased question in the set. It is handled with a `CASE`, not
cleverness.

### Why the same answer can mean opposite things

Surveys do not phrase questions consistently. Compare:

| Question | Answer | What it means |
|---|---|---|
| "Can your site perform cardiac MRI?" | Yes | The site **can** |
| "Do you lack on-site MRI capability?" | Yes | The site **cannot** |

Same word, opposite fact. If you map "Yes" to "available" without reading the
question, you get the second row exactly backwards - and it looks perfectly
clean in the data.

This is **polarity**, and it is the single most likely way an extraction pipeline
produces confident, well-formed, wrong answers. The cell below resolves it by
looking at what was asked, not just what was answered.

In [ ]:
-- RUN. Polarity applied to the Study Assessments question.
SELECT f.document_id,
       f.assessment_question,
       f.assessment_answer,
       CASE WHEN LOWER(f.assessment_question) LIKE '%challenge%'
              OR LOWER(f.assessment_question) LIKE '%difficult%'
             THEN 'NEGATIVE' ELSE 'POSITIVE' END AS polarity,
       CASE
         WHEN LOWER(f.assessment_question) LIKE '%challenge%'
           OR LOWER(f.assessment_question) LIKE '%difficult%'
         THEN IFF(UPPER(f.assessment_answer) = 'YES', 'CANNOT', 'CAN')
         ELSE IFF(UPPER(f.assessment_answer) = 'YES', 'CAN', 'CANNOT')
       END AS capability
FROM (SELECT DISTINCT document_id, assessment_question, assessment_answer
      FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_FLAT) f
ORDER BY polarity, document_id;

Read the last two columns together. Two sites answered **Yes** and it means the
opposite thing in each case, purely because of how the question was worded.

### Score it against ground truth

The scoring scope is each study's **required** items, because that is the actual
business question: can this site run *this* study? A required item that never
appeared in the survey is recorded as unavailable - absence is the finding.

### Scoring against ground truth

Every one of the 10 surveys was read by hand and the correct answers recorded in
`ANALYTICS.GROUND_TRUTH`. That lets us do the thing most extraction projects skip:
measure whether it actually worked.

Two numbers to keep in view:

- **Accuracy** - of the items each study requires, how many did we get right?
- **Confidence** - what did the model itself claim, and does that line up with
  whether it was actually correct?

An unmeasured extraction pipeline is a rumour. This is the cell that turns it
into a claim you can defend.

In [ ]:
-- RUN. Accuracy against ground truth.
CREATE OR REPLACE VIEW AZ_FEASIBILITY_DEMO.ANALYTICS.V_SCORING AS
WITH gt AS (
  SELECT document_id, site_id, study_code, item, category,
         expected_capability, printed_as
  FROM AZ_FEASIBILITY_DEMO.ANALYTICS.GROUND_TRUTH
  WHERE is_required = TRUE
),
pred AS (
  SELECT document_id, canonical_name, capability, confidence,
         name_similarity, raw_item_name
  FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_NORMALISED
)
SELECT gt.document_id, gt.site_id, gt.study_code, gt.item, gt.category,
       gt.expected_capability,
       COALESCE(pred.capability, 'CANNOT') AS predicted_capability,
       gt.printed_as, pred.raw_item_name, pred.name_similarity, pred.confidence,
       (COALESCE(pred.capability, 'CANNOT') = gt.expected_capability) AS is_correct
FROM gt
LEFT JOIN pred ON pred.document_id = gt.document_id
              AND pred.canonical_name = gt.item;

SELECT category,
       COUNT(*)            AS items_scored,
       COUNT_IF(is_correct) AS correct,
       ROUND(100.0 * COUNT_IF(is_correct) / COUNT(*), 1) AS accuracy_pct
FROM AZ_FEASIBILITY_DEMO.ANALYTICS.V_SCORING
GROUP BY category
UNION ALL
SELECT 'TOTAL', COUNT(*), COUNT_IF(is_correct),
       ROUND(100.0 * COUNT_IF(is_correct) / COUNT(*), 1)
FROM AZ_FEASIBILITY_DEMO.ANALYTICS.V_SCORING
ORDER BY category;

### A word on this number

100% is a statement about **these documents**, not about the technique in general.
They were purpose-built to be unambiguous: clean digital text, consistent layouts,
checkboxes drawn as images.

Real survey sets are harder, and it is worth knowing why before you trust a pipeline
like this with a decision. Handwriting, scanned pages, inconsistent question ordering
and marks whose meaning depends on their position all cost accuracy. While building
this lab, three document layouts were tested and one had to be dropped at **50%
accuracy** because it could not be read reliably at all - detail in Appendix A of
your guide.

The transferable part is not the score. It is the **discipline**: score a sample
against ground truth before putting extraction into a decision path, and use the
confidence values to route the doubtful cases to a human.

**Checkpoint 2**

In [ ]:
-- RUN. Should print PASS.
SELECT CASE
         WHEN accuracy >= 0.95 THEN 'PASS - ' || ROUND(accuracy * 100, 1) || '% accuracy on '
                                    || scored || ' required items'
         ELSE 'CHECK - ' || ROUND(accuracy * 100, 1) || '%. Rerun step 2 from the top.'
       END AS checkpoint_2
FROM (SELECT COUNT(*) AS scored,
             DIV0(COUNT_IF(is_correct), COUNT(*)) AS accuracy
      FROM AZ_FEASIBILITY_DEMO.ANALYTICS.V_SCORING);

### Make the rest of the lab use *your* extraction

One thing worth being explicit about, because it is the difference between a demo and
a pipeline.

The two fact tables the app and the agent read - `FACT_CAPABILITY` and
`FACT_SITE_STUDY` - were seeded when your account was built, so that if step 1 or 2
had failed you would still have something to look at. They currently hold the seeded
version.

The cell below rebuilds both **from the rows you just extracted and normalised**.
After it runs, every number in steps 4, 5 and 6 traces back to a PDF that *this*
notebook read.

Note what `FACT_CAPABILITY` does, because it is a real modelling decision rather than
plumbing. Long-form surveys list only what a site **has**; checkbox surveys list every
option, ticked or not. So extraction alone gives you explicit `CANNOT` rows for
checkbox documents and none at all for long-form ones - and "which sites cannot do
Cardiac MRI" would quietly under-report. The table therefore unions two things:
every item actually extracted, plus every item the study **requires** that was not
extracted, recorded as `CANNOT` with source `DERIVED_ABSENT`. The absence is the
finding, and `capability_source` keeps it honest about which is which.

In [ ]:
-- RUN. Rebuilds the two fact tables from your extraction rather than the seed.
--
-- FACT_CAPABILITY = everything extracted, UNION everything required but absent.
-- capability_source distinguishes them so a derived gap is never mistaken for a
-- ticked box.
CREATE OR REPLACE TABLE AZ_FEASIBILITY_DEMO.ANALYTICS.FACT_CAPABILITY AS
WITH extracted AS (
  SELECT
    n.document_id, m.site_id, m.study_code,
    n.canonical_name AS item,
    n.category       AS item_category,
    n.capability,
    n.confidence,
    n.name_similarity,
    n.raw_item_name  AS extracted_as,
    (n.raw_item_name <> n.canonical_name) AS was_normalised,
    'EXTRACTED' AS capability_source
  FROM AZ_FEASIBILITY_DEMO.EXTRACTED.ITEMS_NORMALISED n
  JOIN AZ_FEASIBILITY_DEMO.RAW.DOCUMENT_MANIFEST m ON m.document_id = n.document_id
),
required AS (
  SELECT m.document_id, m.site_id, m.study_code, TRIM(r.value::STRING) AS item
  FROM AZ_FEASIBILITY_DEMO.RAW.DOCUMENT_MANIFEST m
  JOIN AZ_FEASIBILITY_DEMO.RAW.STUDIES st ON st.study_code = m.study_code,
       LATERAL SPLIT_TO_TABLE(st.required_items, '|') r
),
derived_gaps AS (
  SELECT
    rq.document_id, rq.site_id, rq.study_code,
    rq.item,
    t.category AS item_category,
    'CANNOT'   AS capability,
    NULL::FLOAT AS confidence,
    NULL::FLOAT AS name_similarity,
    NULL::VARCHAR AS extracted_as,
    FALSE AS was_normalised,
    'DERIVED_ABSENT' AS capability_source
  FROM required rq
  JOIN AZ_FEASIBILITY_DEMO.ANALYTICS.ITEM_TAXONOMY t ON t.canonical_name = rq.item
  LEFT JOIN extracted e
         ON e.document_id = rq.document_id AND e.item = rq.item
  WHERE e.item IS NULL
),
combined AS (
  SELECT * FROM extracted
  UNION ALL
  SELECT * FROM derived_gaps
)
SELECT
  c.document_id, c.site_id, c.study_code, c.item, c.item_category, c.capability,
  c.confidence, c.name_similarity, c.extracted_as, c.was_normalised,
  c.capability_source,
  -- Only extracted rows can be low confidence. A derived gap has no score, and
  -- flagging it for review would be a false alarm.
  (c.confidence IS NOT NULL AND c.confidence < 0.70) AS needs_review,
  ARRAY_CONTAINS(c.item::VARIANT, SPLIT(st.required_items, '|')) AS is_required_for_study,
  s.institution_name, s.city, s.country, s.region, s.institution_type,
  st.drug_name, st.therapy_area, st.indication, st.phase,
  st.status AS study_status
FROM combined c
JOIN AZ_FEASIBILITY_DEMO.RAW.SITES s    ON s.site_id = c.site_id
JOIN AZ_FEASIBILITY_DEMO.RAW.STUDIES st ON st.study_code = c.study_code;

-- One row per site and study: coverage against what the study requires, plus the
-- enrolment numbers the portfolio page plots against it.
CREATE OR REPLACE TABLE AZ_FEASIBILITY_DEMO.ANALYTICS.FACT_SITE_STUDY AS
SELECT site_id, study_code, patients_seen_per_year, planned_screened, planned_enrolled,
       competing_studies, startup_weeks, items_required, items_available, items_missing,
       capability_coverage, mean_confidence, has_feasibility_survey,
       ARRAY_TO_STRING(ARRAY_COMPACT(missing_items), ', ') AS missing_items
FROM AZ_FEASIBILITY_DEMO.ANALYTICS.V_SITE_STUDY_READINESS;

-- Both sources should be present. EXTRACTED came off a page; DERIVED_ABSENT is a
-- required item the survey never mentioned.
SELECT capability_source,
       COUNT(*) AS row_count,
       COUNT_IF(capability = 'CAN')    AS can_rows,
       COUNT_IF(capability = 'CANNOT') AS cannot_rows
FROM AZ_FEASIBILITY_DEMO.ANALYTICS.FACT_CAPABILITY
GROUP BY capability_source
ORDER BY capability_source;

**Checkpoint 2b**

In [ ]:
-- RUN. Should print PASS.
SELECT CASE
         WHEN cap_rows > 0 AND site_study_rows > 0 AND sources = 2
           THEN 'PASS - ' || cap_rows || ' capability rows across ' || site_study_rows
                || ' site-study pairs, both extracted and derived gaps present'
         WHEN sources < 2
           THEN 'CHECK - only ' || sources || ' capability_source value. Rerun the cell above.'
         ELSE 'CHECK - fact tables look empty. Rerun the cell above.'
       END AS checkpoint_2b
FROM (
  SELECT (SELECT COUNT(*) FROM AZ_FEASIBILITY_DEMO.ANALYTICS.FACT_CAPABILITY) AS cap_rows,
         (SELECT COUNT(*) FROM AZ_FEASIBILITY_DEMO.ANALYTICS.FACT_SITE_STUDY) AS site_study_rows,
         (SELECT COUNT(DISTINCT capability_source)
            FROM AZ_FEASIBILITY_DEMO.ANALYTICS.FACT_CAPABILITY) AS sources
);

---

# Step 3: Parse, chunk and search

## Extraction is not the whole document

Step 1 pulled out the fields you knew you wanted. But a feasibility survey also
contains free text nobody can schematise in advance - the comments box where a site
explains *why* something is unavailable, or flags a screening threshold it is unsure
about.

That text is where the reasons live. You cannot get at it with a fixed schema, so you
need the other half of the pipeline: parse the document to text, split it into
retrievable pieces, and index it.

`AI_PARSE_DOCUMENT` is the counterpart to `AI_EXTRACT`. Same file, different job:

| | Returns | Use for |
|---|---|---|
| `AI_EXTRACT` | Structured JSON against your schema | Fields you can name in advance |
| `AI_PARSE_DOCUMENT` | The document as markdown text | Everything else |

`mode => 'LAYOUT'` preserves structure - headings, lists and tables come back as
markdown rather than a flat wall of text, which makes the chunks far more coherent.
`page_split => TRUE` returns one entry per page instead of one long string.

One detail worth watching for in the output: on the checkbox surveys, `LAYOUT` mode
brings the tick state back as **checked and unchecked box characters**. The parse keeps
the visual state, it does not silently drop it. That is why parsing is a genuine
alternative route to the same answer rather than a lossy fallback.

### Keeping the words

Steps 1 and 2 gave us numbers - which items a site can and cannot do. But the
reason a site is difficult almost never appears in a tick box. It appears in a
comments field at the bottom of page 2.

`AI_PARSE_DOCUMENT` returns the document as markdown. Two options matter:

- **`mode: LAYOUT`** preserves structure - headings, tables, and critically the
  checkbox characters themselves, so a ticked box arrives as a character you can
  read rather than being dropped.
- **`page_split: TRUE`** returns one entry per page instead of one wall of text,
  which is what lets a search result cite a page number.

This is the same PDF as step 1, read a second way. Extraction gave us the facts;
parsing keeps the language they were written in.

In [ ]:
-- RUN. One document, so you can see what the parser returns. About 4 seconds.
--
-- mode LAYOUT keeps the structure of the page: headings stay headings, tables stay
-- tables, and a ticked checkbox is still legible as a ticked checkbox.
-- page_split TRUE returns one entry per page rather than one wall of text, which is
-- what lets a search result cite a page number later.
SELECT pg.index                 AS page_index,
       LEFT(pg.value:content::STRING, 900) AS page_markdown
FROM (
  SELECT AI_PARSE_DOCUMENT(
           TO_FILE('@AZ_FEASIBILITY_DEMO.RAW.SURVEYS',
                   'survey_01_site_001_d0000c00010.pdf'),
           {'mode': 'LAYOUT', 'page_split': TRUE}) AS payload
) p, LATERAL FLATTEN(input => p.payload:pages) pg
ORDER BY page_index;

That is the same PDF step 1 extracted from, read a second way. Extraction gave us the
facts; parsing keeps the language they were written in.

The next cell does the same for all ten surveys, and it is the **longest wait in the
lab**. Parsing is charged per document and runs largely one document at a time, so
ten documents take much longer than one - several minutes rather than seconds.

The cell takes the warehouse up to 2X-Large for the duration and puts it straight
back afterwards. That is worth understanding rather than copying blindly: for
document AI work, **warehouse size buys you throughput across files, not speed on
any single file**. Ten documents can only spread so far, so a bigger warehouse helps
less than the size difference suggests - and a hundred documents would benefit far
more than ten do. Sizing up around a known burst and dropping straight back is the
pattern; expecting size alone to make one document fast is the mistake.

In [ ]:
-- RUN. All 10 surveys to markdown. This is the longest wait in the lab - expect a
-- few minutes - so it is worth reading the notes above while it runs.
--
-- Scale up for the parse, then straight back down. Nothing else is running, so
-- the extra size costs nothing beyond the seconds it is actually held.
ALTER WAREHOUSE AZ_HOL_WH SET WAREHOUSE_SIZE = XXLARGE;

CREATE OR REPLACE TABLE AZ_FEASIBILITY_DEMO.EXTRACTED.PAGES AS
WITH parsed AS (
  SELECT REGEXP_REPLACE(relative_path, '\\.pdf$', '') AS document_id,
         AI_PARSE_DOCUMENT(
           TO_FILE('@AZ_FEASIBILITY_DEMO.RAW.SURVEYS', relative_path),
           {'mode': 'LAYOUT', 'page_split': TRUE}
         ) AS payload
  FROM DIRECTORY(@AZ_FEASIBILITY_DEMO.RAW.SURVEYS)
)
SELECT document_id,
       pg.index                        AS page_index,
       pg.value:content::STRING        AS page_text,
       payload:metadata:pageCount::INT AS page_count
FROM parsed, LATERAL FLATTEN(input => payload:pages) pg;

ALTER WAREHOUSE AZ_HOL_WH SET WAREHOUSE_SIZE = MEDIUM;

SELECT COUNT(*) AS pages,
       COUNT(DISTINCT document_id) AS documents,
       ROUND(AVG(LENGTH(page_text))) AS avg_chars_per_page
FROM AZ_FEASIBILITY_DEMO.EXTRACTED.PAGES;

In [ ]:
-- RUN. Read the free text the extraction step never touched.
-- This is page 2 of the Pune survey. Look at the Additional Comments section.
SELECT page_index, page_text
FROM AZ_FEASIBILITY_DEMO.EXTRACTED.PAGES
WHERE document_id = 'survey_07_site_007_d0000c00030'
  AND page_index = 1;

Notice what is in there: *"Our site does not currently have cardiac mri scan available
on site. We would need to refer to a third party facility, which may extend the
screening window beyond the protocol requirement."*

Step 1 gave you `Cardiac MRI = CANNOT`. This gives you the **reason**, and a
consequence nobody asked about - a possible protocol deviation on screening windows.
No fixed schema would have found that.

## Chunking

You cannot index a whole page and expect precise retrieval. Split it.

Two parameters matter:

- **Chunk size.** Too large and a hit returns mostly irrelevant text. Too small and
  you cut sentences in half and lose the context that made them meaningful.
- **Overlap.** Chunks share a margin so a sentence spanning a boundary still appears
  whole in at least one chunk.

`SPLIT_TEXT_RECURSIVE_CHARACTER` with `'markdown'` splits on structure first -
headings, then paragraphs, then sentences - and only falls back to cutting mid-text
if it has to. That is why parsing in `LAYOUT` mode earlier pays off here.

### Why documents get chopped up

You cannot search a whole document usefully. Ask "which sites are worried about
staffing" against a full survey and the answer is drowned by four pages of
unrelated tick boxes.

So the text is split into **chunks**, and each chunk is indexed separately.
`SPLIT_TEXT_RECURSIVE_CHARACTER` splits on structure first - headings, then
paragraphs, then sentences - so it breaks at natural boundaries rather than
mid-word.

Two numbers control it, and they matter more than they look:

- **Chunk size, 400 characters.** Too large and the comments box shares a chunk
  with the capability list, so a search for a concern returns a wall of tick
  boxes. Too small and sentences get cut in half.
- **Overlap, 80 characters.** Consecutive chunks share a little text so a
  sentence spanning a boundary is still findable from either side.

At 400 the comments section gets a chunk of its own. That single choice is why the
search below works.

In [ ]:
-- RUN. Chunks the pages and joins the metadata a searcher will want to filter on.
-- 400 characters with 80 of overlap. See the note above for why that size.
CREATE OR REPLACE TABLE AZ_FEASIBILITY_DEMO.ANALYTICS.SURVEY_CHUNKS AS
WITH chunked AS (
  SELECT p.document_id, p.page_index,
         c.index         AS chunk_index,
         c.value::STRING AS chunk_text
  FROM AZ_FEASIBILITY_DEMO.EXTRACTED.PAGES p,
       LATERAL FLATTEN(input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
         p.page_text,
         'markdown',
         400,   -- chunk size in characters
         80     -- overlap between consecutive chunks
       )) c
)
SELECT c.document_id, c.page_index, c.chunk_index, c.chunk_text,
       m.site_id, m.study_code,
       s.institution_name, s.city, s.country, s.region,
       st.drug_name, st.therapy_area, st.indication, st.phase,
       -- Prefixing each chunk with who and what it is about means a search for
       -- "Pune cardiac imaging" can match on the metadata as well as the body.
       'Site ' || m.site_id || ' (' || s.institution_name || ', ' || s.city
         || ', ' || s.country || ') - study ' || m.study_code || ' '
         || st.drug_name || ' for ' || st.indication || CHR(10) || CHR(10)
         || c.chunk_text AS search_text
FROM chunked c
JOIN AZ_FEASIBILITY_DEMO.RAW.DOCUMENT_MANIFEST m ON m.document_id = c.document_id
JOIN AZ_FEASIBILITY_DEMO.RAW.SITES   s  ON s.site_id    = m.site_id
JOIN AZ_FEASIBILITY_DEMO.RAW.STUDIES st ON st.study_code = m.study_code;

SELECT COUNT(*) AS chunks,
       COUNT(DISTINCT document_id) AS documents,
       ROUND(AVG(LENGTH(chunk_text))) AS avg_chunk_chars,
       MIN(LENGTH(chunk_text)) AS smallest,
       MAX(LENGTH(chunk_text)) AS largest
FROM AZ_FEASIBILITY_DEMO.ANALYTICS.SURVEY_CHUNKS;

### What a Cortex Search service is

A search service that understands meaning, created with one DDL statement. No
vector database to stand up, no embedding pipeline to schedule, no separate
service to keep in sync.

Three parts of the definition do the work:

- **`ON`** is the column that gets indexed and searched.
- **`ATTRIBUTES`** are columns you can filter by at query time - site, study,
  page - so you can scope a search rather than always hitting everything.
- **`TARGET_LAG`** is how fresh the index stays. Set it and Snowflake keeps the
  index current as the underlying table changes; there is no refresh job to own.

One gotcha worth naming: any column you want **returned** has to be in the
`AS SELECT`. Listing it under `ATTRIBUTES` alone is not enough, and the error when
you get it wrong is blunt - *"column was not indexed in this Cortex Search
Service"*.

In [ ]:
-- RUN. Builds the search service. Returns in about 10 seconds, then indexes in the
-- background for roughly a minute.
--
-- ON names the column that gets indexed for semantic search.
-- ATTRIBUTES names the columns you can filter on at query time.
CREATE OR REPLACE CORTEX SEARCH SERVICE AZ_FEASIBILITY_DEMO.ANALYTICS.SURVEY_SEARCH
  ON search_text
  ATTRIBUTES document_id, site_id, study_code, institution_name, city, country,
             region, therapy_area, phase, page_index
  WAREHOUSE = AZ_HOL_WH
  TARGET_LAG = '1 hour'
  COMMENT = 'Chunked feasibility survey text.'
  AS SELECT search_text, chunk_text, document_id, site_id, study_code,
            institution_name, city, country, region, therapy_area, phase, page_index
     FROM AZ_FEASIBILITY_DEMO.ANALYTICS.SURVEY_CHUNKS;

SHOW CORTEX SEARCH SERVICES IN SCHEMA AZ_FEASIBILITY_DEMO.ANALYTICS;

### Try it

`TARGET_LAG = '1 hour'` means the index refreshes itself when the underlying table
changes - you do not rebuild it by hand when a new survey lands.

Give it about a minute to finish its first index, then run the next cell. If you get
no results, wait and run it again.

### What the next cell proves

It asks the index a question in the words a person would use: *"we would need to refer
patients to a third party facility"*.

**That sentence does not appear in any document.** No survey contains the phrase "refer
patients". So a keyword search returns nothing at all. The cell below returns the sites
whose comments say exactly that in their own words - and tells you which page to look
at to check.

Read the result as: which sites raised this concern, and where do I verify it.

In [ ]:
-- RUN. Semantic search over the surveys. The question below is deliberately worded
-- the way a person would ask it, and appears verbatim in none of the documents.
SELECT ROW_NUMBER() OVER (ORDER BY 1)  AS best_match_rank,
       r.value:site_id::STRING         AS site,
       r.value:city::STRING            AS city,
       'page ' || (r.value:page_index::INT + 1) AS found_on,
       r.value:chunk_text::STRING      AS what_the_site_wrote
FROM TABLE(FLATTEN(input => PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'AZ_FEASIBILITY_DEMO.ANALYTICS.SURVEY_SEARCH',
    '{"query": "we would need to refer patients to a third party facility",
      "columns": ["site_id","city","chunk_text","page_index"],
      "limit": 4}'
  )):results)) r;

Every hit is an **Additional Comments** section - free text, not tick
boxes. That is the chunk size doing its job: at 400 characters the comments box gets a
chunk of its own, so a question about a concern retrieves the concern.

Nothing in that question appears verbatim in the documents. It matched on meaning.

**Over to the room.** Someone give me a concern to search for, in your own words -
*"worried about the freezer"*, *"lab is off site"*, *"not enough patients"*. I will
put it in and we will see what comes back.

**Checkpoint 3**

In [ ]:
-- RUN. Should print PASS.
SELECT CASE
         WHEN pages = 20 AND chunks BETWEEN 60 AND 140
           THEN 'PASS - ' || pages || ' pages parsed into ' || chunks || ' chunks'
         ELSE 'CHECK - ' || pages || ' pages, ' || chunks || ' chunks. Rerun step 3 from the top.'
       END AS checkpoint_3
FROM (SELECT (SELECT COUNT(*) FROM AZ_FEASIBILITY_DEMO.EXTRACTED.PAGES) AS pages,
             (SELECT COUNT(*) FROM AZ_FEASIBILITY_DEMO.ANALYTICS.SURVEY_CHUNKS) AS chunks);

---

# Step 4: Semantic view

Step 3 made the free text searchable. This step makes the **numbers** askable.

A semantic view is the contract between your tables and natural language: what the
entities are, how they join, which measures exist, and what the business rules are.

It is already built in your account, because the agent in the next step needs it warm.
It defines 4 logical tables, 4 relationships, 10 facts, 22 dimensions and 11 metrics.

### What is actually in it

A semantic view has four kinds of thing in it, and the distinction matters:

- **Logical tables** - the entities, named as the business names them. A `SITE` is
  a site, not `DIM_SITE`.
- **Relationships** - how they join, defined once. Nobody rediscovers the join key
  and nobody gets it wrong.
- **Facts and dimensions** - the raw columns, with synonyms. `startup_weeks` also
  answers to "time to first patient".
- **Metrics** - the calculations, defined centrally. This is where "capability
  coverage" stops being something everyone works out for themselves.

The cell below counts each. Then we will open the model itself.

In [ ]:
-- RUN. The shape of the model in one row.
DESCRIBE SEMANTIC VIEW AZ_FEASIBILITY_DEMO.SEMANTIC_VIEWS.SITE_FEASIBILITY;

In [ ]:
-- RUN. Summarises the DESCRIBE output above into a single row.
-- DESCRIBE returns its column names quoted and lowercase, hence the quoting.
SELECT
  COUNT(DISTINCT CASE WHEN "object_kind" = 'TABLE'        THEN "object_name" END) AS business_entities,
  COUNT(DISTINCT CASE WHEN "object_kind" = 'RELATIONSHIP' THEN "object_name" END) AS joins_defined,
  COUNT(DISTINCT CASE WHEN "object_kind" = 'DIMENSION'    THEN "object_name" END) AS attributes,
  COUNT(DISTINCT CASE WHEN "object_kind" = 'FACT'         THEN "object_name" END) AS measurable_columns,
  COUNT(DISTINCT CASE WHEN "object_kind" = 'METRIC'       THEN "object_name" END) AS governed_metrics
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

The number to point at is **governed metrics**. Those calculations are defined once, in
the model, so "study requirements met" is the same number in a notebook, in the app and
in an agent's answer - not one number per analyst.

### Open the model as a file, and deploy your own copy

In the file list on the left of this workspace there is
**`site_feasibility_test.sv.yaml`**. Open it. The `.sv.yaml` extension tells Snowsight
this is a semantic model, so it opens in the **Semantic View Editor** rather than as a
text file, with a **Visual** and a **YAML** toggle.

First point, and it is the important one: **the model is a text file**. Every table,
join, synonym, metric and business rule in one reviewable artefact. It goes into version
control, it gets code reviewed, and it moves between environments like anything else. A
semantic model is not something you click together once and hope nobody changes it.

You will see *"Unable to validate. Deploy to see validation"* and *"No live semantic
views available"*. That is correct: the file is a **draft**, and validation and the
Playground both need a live object to run against.

So deploy it.

**Click Deploy.** Target name **`SITE_FEASIBILITY_TEST`**, into
**`AZ_FEASIBILITY_DEMO.SEMANTIC_VIEWS`**. Validation and the Playground now light up and
the model is yours to play with.

**Why a separate name.** Your account already has a deployed `SITE_FEASIBILITY`, and the
agent in step 5 is wired to it. Deploying under a different name gives you a sandbox
copy and leaves the agent's model untouched. Worth knowing how this actually works: the
name comes from the **`name:` field at the top of the YAML**, not from the dialog box.
Deploying a file whose `name:` matches an existing view replaces that view in place -
`CREATE OR REPLACE SEMANTIC VIEW ... COPY GRANTS`, so grants survive, but the definition
is overwritten. This file is already named `SITE_FEASIBILITY_TEST`, so it cannot touch
the live one.

**Four things to look at in your copy.**

1. **Logical tables.** Four entities, each mapped to a physical table. The model is a
   layer over the warehouse, not a copy of it.
2. **Relationships.** The joins, declared once. This is what stops an agent inventing a
   join and quietly double counting.
3. **Metrics.** Open `m_capability_coverage`. It is a `SUM` over a `SUM`, never an
   average of percentages - because averaging weights a site with 7 required items the
   same as one with 10, which is quietly wrong and very hard to spot on a dashboard.
4. **Custom instructions.** Where the business rules live, travelling with the model.
   The important one: **not assessed is not the same as not capable.** Only 10 of the 40
   sites returned a survey, so a site without one has a gap in our knowledge, not a gap
   in its capability. Conflating those two makes the portfolio look far worse than it is.

Then use the **Playground**: ask one question in plain language and read the SQL it
generates. English in, governed SQL out, against your joins and your metric definitions.

Your copy is identical to the model the agent uses, so what you see here is what the
agent sees in step 5. The last cell in this notebook drops the copy again.

In [ ]:
-- RUN. Query the semantic view directly.
SELECT * FROM SEMANTIC_VIEW(
  AZ_FEASIBILITY_DEMO.SEMANTIC_VIEWS.SITE_FEASIBILITY
  METRICS site_study.m_capability_coverage,
          site_study.m_planned_enrolled,
          site_study.m_items_missing
  DIMENSIONS sites.site, sites.city, sites.country
  WHERE site_study.f_items_required > 0
) ORDER BY m_capability_coverage DESC, m_planned_enrolled DESC;

### One more question, no new SQL

The point of defining metrics centrally is that new questions do not need new
logic. Same view, different dimension - a country rollup, using the same coverage
metric with no recalculation.

Note the filter: only countries with assessed sites appear. Without it, every
country with no survey would show as 0% coverage, which would read as incapable
when it actually means unmeasured - exactly the confusion the model's instructions
tell it to avoid.

In [ ]:
-- RUN. The same metrics, rolled up by country instead of by site.
-- Coverage is recomputed at this grain, not averaged from the site numbers.
SELECT * FROM SEMANTIC_VIEW(
  AZ_FEASIBILITY_DEMO.SEMANTIC_VIEWS.SITE_FEASIBILITY
  METRICS site_study.m_capability_coverage,
          site_study.m_planned_enrolled,
          site_study.m_items_missing
  DIMENSIONS sites.country
  WHERE site_study.f_items_required > 0
) ORDER BY m_capability_coverage DESC;

**Checkpoint 4**

In [ ]:
-- RUN. Should print PASS.
SELECT CASE
         WHEN COUNT(*) = 10 AND MAX(coverage) = 1.0
           THEN 'PASS - semantic view returns ' || COUNT(*) || ' assessed sites, '
                || 'best coverage ' || ROUND(MAX(coverage) * 100) || '%'
         ELSE 'CHECK - got ' || COUNT(*) || ' assessed sites'
       END AS checkpoint_4
FROM (
  SELECT * FROM SEMANTIC_VIEW(
    AZ_FEASIBILITY_DEMO.SEMANTIC_VIEWS.SITE_FEASIBILITY
    METRICS site_study.m_capability_coverage
    DIMENSIONS sites.site
    WHERE site_study.f_items_required > 0
  ) AS t(coverage, site)
);

---

# Step 5: An agent with three tools

The agent in your account has three tools and decides for itself which to use:

| Tool | For |
|---|---|
| `feasibility_analyst` | Anything countable. Cortex Analyst over the semantic view. |
| `survey_search` | What a site actually *wrote*. Cortex Search over the survey text. |
| `code_execution` | Python in a sandbox, for charts and composite scoring. |

It is one object holding all three. There is no orchestration loop to write, no
retriever to tune, and no second copy of the data.

### What an agent adds

The semantic view answers questions about numbers. The search service answers
questions about words. Most real questions need both:

> *"Shortlist sites for the cardiology study and tell me what the investigators
> are worried about."*

The shortlist is a SQL question. The worries are in free text. An **agent** is
given both tools and decides which to use for which part of the question, then
writes the answer.

Two things in its definition are worth knowing:

- **Tools** - here, the semantic view, the search service, and a Python sandbox
  for arithmetic it should not be inventing in SQL.
- **Instructions** - plain English rules about how to behave. This is where you
  say "always name the site" or "never average a coverage ratio". It is the
  cheapest place to encode institutional knowledge.

The agent is already built. The next cell confirms it exists; then we will use it
from the app in step 6, which is where a business user would actually meet it.

In [ ]:
-- RUN. Confirm the agent and its tools.
SHOW AGENTS IN SCHEMA AZ_FEASIBILITY_DEMO.ANALYTICS;

### Look at the agent before you talk to it

Before asking it anything, open the definition. In the left navigation choose
**AI & ML** then **Agents**, and open **SITE_FEASIBILITY_AGENT**.

An agent is a configured object, not a chat window. Three parts are worth reading:

- **Tools.** This one has three: the semantic view for numbers, Cortex Search for the
  free text, and a Python sandbox for charts. It picks between them per question.
- **Instructions.** Plain language, and they are doing real work - including the rule
  that a site with no survey is unassessed rather than incapable.
- **Access.** It runs as the querying user, so it can only see what that user can see.
  There is no second copy of the data and no separate permission model to keep in step.

That is the whole thing. No orchestration code, no vector store to provision, no
retrieval logic to maintain.

### The tools you did not use

This agent has three tools because three is what the lab needs. The list you can
choose from is longer, and it is worth knowing what is on it before you design your
own:

| Tool | What it is |
|---|---|
| **Cortex Analyst** | Natural language to SQL over a semantic view. Numbers. |
| **Cortex Search** | Retrieval over unstructured text. Words. |
| **Code execution** | Python in an isolated sandbox. See below. |
| **Data to Chart** | Turns results from another tool into a visualisation. |
| **Custom tools** | Your own stored procedures and UDFs, so the agent can call your business logic or a backend system. |
| **Agent skills** | A packaged bundle of instructions plus scripts on a stage, giving the agent a repeatable task-specific capability. Reusable across agents. |
| **MCP connectors** | Tools hosted on a remote **Model Context Protocol** server - Jira, Salesforce, or your own service. This is how an agent reaches outside Snowflake. |
| **Web search** | The public internet. Off by default and enabled per account. |

**Code execution, since it is the one people misread.** It runs Python in a secure
sandbox with `pandas`, `numpy`, `matplotlib` and `plotly` already there, and it
**cannot query your data**. When the agent needs rows it runs SQL with its SQL tools,
outside the sandbox, and hands the results in. So it is the right tool for medians,
weighted scores and charts, and the wrong tool for anything that needs a fresh query.
Extra PyPI packages come from the Artifact Repository, not from `pip` inside the box.

Two ways it fails quietly, both worth carrying into your own build:

- **Owner's rights invocation strips it.** Call the agent from an owner's rights
  stored procedure and no sandbox is created, the tool is removed from the request,
  and the agent answers in prose instead. It looks like the model chose not to write
  code. Invoke with caller's rights if it needs the sandbox.
- **`code_toolset_all` and `code_execution` are mutually exclusive.** The first is
  the full Cortex Code toolset - bash, file editing, grep, SQL, skills - for an
  autonomous coding agent. Asking for both returns an error.

### Now put it to work in CoWork

**First, add the agent to CoWork.** On the agent page you are already looking at,
click **+ Add to Snowflake CoWork** in the header. Until you do that the agent
exists but does not appear in the CoWork picker, which reads as a broken deployment
and is not one.

Then open **CoWork** from the left navigation (the sparkle icon) and select the
**Site Feasibility Assistant**.

Ask these four in order. Each one is designed to pull a different tool.

**1. Cortex Search** - what a site said, not what it can do

> Which sites raised concerns in their survey comments, and what exactly did they write?

**2. Cortex Analyst** - structured, countable

> Which sites can perform Cardiac MRI, and which cannot?

This is the question that closes the loop, so do not rush past it. Every row behind
that answer came out of a **PDF**. `AI_EXTRACT` in step 1 read the surveys,
step 2 resolved the item names into `EXTRACTED.ITEMS_NORMALISED`, and
`ANALYTICS.FACT_CAPABILITY` sits on top of that - which is what the semantic view
counts. Nobody typed a capability into a table.

Look at what comes back for Cardiac MRI: one site **CAN** and one **CANNOT**, and
the two rows do not have equal standing. The CAN is `EXTRACTED` - a box was ticked
on a survey and the model read it. The CANNOT is `DERIVED_ABSENT` - the study
requires Cardiac MRI and the survey never mentioned it, so the gap is inferred from
silence. Both are true and only one is evidence. That distinction is carried in
`capability_source` precisely so an agent can never quietly blur them.

**3. Code execution** - this one has to run Python

> Plot capability coverage against planned enrolment for all assessed sites and highlight any site that is high enrolment but low coverage.

**4. Your own question.** Anything you like.

After each answer, expand the **reasoning** block and click **Show SQL**. You are
looking at the tool choice being made. On question 3 you should see it retrieve the
numbers with SQL, then write and run Python to compute the medians and flag the
quadrant, then render the chart. That is three tools in one answer.

Expected on question 3: **SITE_007 Pune** and **SITE_005 Krakow** flagged - both
above median enrolment at 67% coverage.

In [ ]:
-- RUN (optional). The same question from SQL, if you would rather not leave the notebook.
-- Note the payload must be a JSON STRING, not a SQL object.
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'AZ_FEASIBILITY_DEMO.ANALYTICS.SITE_FEASIBILITY_AGENT',
  '{"messages": [{"role": "user", "content": [{"type": "text",
     "text": "Which sites can perform Cardiac MRI, and which cannot?"}]}]}'
) AS agent_response;

**Checkpoint 5** - you have asked the agent at least one question and seen it choose
a tool. Nothing to run.

---

# Step 6: Deploy the app

Everything so far has been SQL. This is where it becomes something you could hand
to a study start-up lead who has never opened a worksheet.

The app files are already staged. You run one statement and Snowflake builds and
runs the container.

In [ ]:
-- RUN. The app files waiting on your stage.
SELECT RELATIVE_PATH AS file, ROUND(SIZE / 1024, 1) AS size_kb
FROM DIRECTORY(@AZ_FEASIBILITY_DEMO.STREAMLIT.APP_FILES)
ORDER BY RELATIVE_PATH;

In [ ]:
-- RUN. Your deploy.
--
-- FROM, not ROOT_LOCATION. ROOT_LOCATION is legacy and cannot use the container
-- runtime at all, so it is the wrong choice here even though you will still see it
-- in older examples.
--
-- FROM copies the files into a versioned stage inside the Streamlit object at create
-- time. That means if you edit app.py on the stage later, you rerun this cell to pick
-- the change up - the app does not track the stage.
CREATE OR REPLACE STREAMLIT AZ_FEASIBILITY_DEMO.STREAMLIT.MY_SITE_SHORTLIST
  FROM '@AZ_FEASIBILITY_DEMO.STREAMLIT.APP_FILES'
  MAIN_FILE = 'app.py'
  QUERY_WAREHOUSE = AZ_HOL_WH
  TITLE = 'Site Feasibility'
  RUNTIME_NAME = 'SYSTEM$ST_CONTAINER_RUNTIME_PY3_11'
  COMPUTE_POOL = AZ_HOL_POOL
  EXTERNAL_ACCESS_INTEGRATIONS = (AZ_HOL_PYPI_ACCESS);

-- A newly created app is not live until you either visit it as the owning role or
-- do this. Without it your first click can land on a blank app.
ALTER STREAMLIT AZ_FEASIBILITY_DEMO.STREAMLIT.MY_SITE_SHORTLIST
  ADD LIVE VERSION FROM LAST;

SHOW STREAMLITS LIKE 'MY_SITE_SHORTLIST' IN SCHEMA AZ_FEASIBILITY_DEMO.STREAMLIT;

### While that builds

The first deploy takes a couple of minutes, and it is worth knowing what it is
doing rather than watching a spinner.

**Two runtimes, one object.** A Streamlit app in Snowflake can run on your
warehouse or in a container. Warehouse runtime starts fast and draws packages from
the Snowflake Anaconda channel. **Container runtime** gives you a real Python
environment on a compute pool - any PyPI package, more memory, and Streamlit
features the warehouse runtime does not carry. This app needs one of those.
The Document review page shows the survey PDF beside what was extracted from it,
and it renders that PDF itself, server side, with **pypdfium2** - a PyPI wheel
that the Anaconda channel does not carry. That single dependency is why we are on
containers.

**Three lines in the statement above are doing the work:**

- **`COMPUTE_POOL`** - the container needs somewhere to run. A pool is a managed set
  of nodes; this one is shared and already awake, which is why you are waiting two
  minutes and not five.
- **`EXTERNAL_ACCESS_INTEGRATIONS`** - container runtime resolves dependencies from
  **PyPI**, not the Anaconda channel, so it needs egress. Egress in Snowflake is
  never implicit: you define a network rule listing exactly which hosts are
  reachable, wrap it in an integration, and grant that. Without it the build fails
  with a DNS error, which looks like a network fault and is actually a governance
  control doing its job.

And the dependencies come from **`pyproject.toml`** on the stage, not
`environment.yml` - the container runtime reads one and the warehouse runtime reads
the other. Ours asks for `pypdfium2`, which is what renders the survey pages.
One quiet gotcha in that file: leave the dependency list empty and the build fails
outright, because the base image does not include Streamlit itself.
- **`FROM`, not `ROOT_LOCATION`** - `FROM` copies the files into a versioned stage
  inside the Streamlit object. `ROOT_LOCATION` is legacy and cannot use the
  container runtime at all, so older examples you find will not work here.

**And `ADD LIVE VERSION FROM LAST`.** A newly created app has a version but no live
one. Skip that line and your first click lands on a blank app.

### Open it

**Projects** → **Streamlit** → `MY_SITE_SHORTLIST`.

Click through all four pages:

- **Portfolio** - the shortlist, requirements met by category, and the
  coverage-against-enrolment scatter. The upper-left of that scatter is the risk
  quadrant: enrolment ambition without the capability to deliver it.
- **Document review** - the survey PDF on the left, the rows the model produced from
  it on the right, with a confidence score per field.
- **Site detail** - pick SITE_007 Pune. Every capability, its confidence, the name as
  it was written on the survey, and the similarity score that resolved it. Note the
  banner about gaps inferred from absence.
- **Ask** - the same agent from step 5, embedded in the app.

### Four things to look at

**1. The portfolio, and why nothing is at 100%.**
The headline is how many of each study's required items each site can actually
provide. Requirements are not a fixed list - each study declares its own in
`RAW.STUDIES.REQUIRED_ITEMS`, so the same site is measured differently depending
on what is being asked of it. Only one site in the portfolio meets every
requirement of its study.

**2. SITE_009, New Delhi - 6 of 7.**
Top of the shortlist on score, and still short one item. Open it. The missing item
is X-Ray, and the evidence line says `EXTRACTED` - the survey was asked and said
no. That is a known gap, not a data problem. Contrast it with a `DERIVED_ABSENT`
row, where the study requires something the survey never mentioned. Both count
against the site; only one of them is a fact the site told you.

**3. The score is a judgement, not a standard.**
It combines requirements met, planned enrolment and startup speed. Those weights
are a choice about what your programme values, and reasonable people would set
them differently. Weight requirements heavily and SITE_001 Chennai goes to the
top, because it is the only site meeting every requirement. Weight enrolment and
New Delhi wins on volume despite the gap. The card on the page says which weights
are in force - the ranking is arguable, and it should be visible enough to argue
with.

**4. Document review - the trust page.**
The PDF on the left, what the model extracted from it on the right. This is the
page that decides whether anyone believes the other three. A business user does
not want an accuracy metric; they want to see the ticked box and the row it
produced, side by side, for a document they recognise. Everything upstream in this
lab exists to make this page true.

Then use **Ask** for the question that needs both halves: *"shortlist sites for the
cardiology study and tell me what the investigators are worried about"*. The
shortlist comes from the semantic view, the worries from the search service, in one
answer.

**Checkpoint 6**

In [ ]:
-- RUN. Should print PASS.
SHOW STREAMLITS LIKE 'MY_SITE_SHORTLIST' IN SCHEMA AZ_FEASIBILITY_DEMO.STREAMLIT;

In [ ]:
-- RUN. Reads the result of the cell above.
SELECT CASE WHEN COUNT(*) > 0
              THEN 'PASS - your app is deployed on ' ||
                   COALESCE(MAX("query_warehouse"), 'unknown warehouse')
            ELSE 'CHECK - MY_SITE_SHORTLIST not found. Rerun the deploy cell above.'
       END AS checkpoint_6
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

---

# What you built

Starting from 10 PDFs on a stage:

1. **Extracted** structured capability data with a single `AI_EXTRACT` call reading
   the page image directly, with a calibrated confidence score on every field.
2. **Normalised** free-text item names against a controlled vocabulary with one
   deterministic embedding join, and derived capability while respecting question
   polarity.
3. **Scored** the result against ground truth, so the pipeline's accuracy is a
   number rather than a hope.
4. **Modelled** it as a semantic view carrying the business rules, so coverage is
   never averaged and unassessed is never confused with incapable.
5. **Asked** an agent that chose between structured query, document search and
   writing its own Python.
6. **Deployed** an app on Snowpark Container Services and changed the judgement
   encoded in its ranking.

What this replaces is a person reading PDFs and typing into a spreadsheet, for weeks
per study, with no audit trail back to the source page.

Every number in the app traces to an extracted field, with a confidence score and
the original wording preserved. When somebody asks *"why is Pune at 67%?"*, the
answer is three clicks away, not a re-read of the PDF.

### Where to take it

- **Fine-tune `arctic-extract`** on your own document set. The layouts that fail
  hardest are the ones worth training on.
- **Route low-confidence extractions to human review** rather than treating all
  output as equal. The threshold in this lab is 0.70 and it is evidence based.
- **Add the survey corpus to Cortex Search across studies**, so a question like
  "which sites have ever mentioned freezer capacity" spans your whole history.
- **Put the pipeline behind a stream and a task** so a new survey landing on the
  stage updates the portfolio without anyone running a notebook.

### Optional: clean up

Only if you want to reset. This drops the objects you created, not the source data.

In [ ]:
-- Drops the sandbox semantic view you deployed in step 4. The model the agent
-- uses, SITE_FEASIBILITY, is a different object and is left alone.
DROP SEMANTIC VIEW IF EXISTS AZ_FEASIBILITY_DEMO.SEMANTIC_VIEWS.SITE_FEASIBILITY_TEST;

-- RUN only if you want to reset. Set to TRUE first.
SET confirm_cleanup = FALSE;

EXECUTE IMMEDIATE $$
BEGIN
  IF ($confirm_cleanup) THEN
    DROP STREAMLIT IF EXISTS AZ_FEASIBILITY_DEMO.STREAMLIT.MY_SITE_SHORTLIST;
    RETURN 'Cleaned up.';
  END IF;
  RETURN 'Nothing dropped. Set confirm_cleanup to TRUE if you meant to.';
END;
$$;